# Predicting food inflation in Turkey

**Question:** Do exchange rates and global food prices help predict monthly food inflation in Turkey?

We will compare Turkish food-price changes with USD/TRY changes and global food-price changes, including related food groups.

The cell below downloads raw data into `data/raw/`. Turkish CPI and USD/TRY are requested from January 2005 through the last completed month. It does not clean or merge the data.

- **Turkish food CPI and food categories:** [TÜİK](https://veriportali.tuik.gov.tr/tr/press/58289), downloaded through [TCMB EVDS](https://evds3.tcmb.gov.tr/). CPI is a price index; 2025=100 is its reference scale.
- **USD/TRY exchange rate:** [TCMB EVDS](https://evds3.tcmb.gov.tr/), daily buying rates.
- **Global food prices:** [FAO Food Price Index](https://www.fao.org/worldfoodsituation/foodpricesindex/en/), monthly overall and food-group indices. The CSV includes its full available history.

Install `requests` and `python-dotenv` in the notebook's Python environment with `pip install requests python-dotenv` if needed. To get the free API key, open [EVDS](https://evds3.tcmb.gov.tr/), choose “Benim Sayfam / My Page”, register if needed, and sign in. Open your username menu, choose “Profilim / Profile”, and select “API Anahtarını Kopyala / Copy API Key” near the bottom. Paste the key after `EVDS_API_KEY=` in the local `.env` file and save it. The cell reads that file automatically and uses a hidden prompt if the key is still empty. The `.env` file is excluded from Git. Do not paste the key into the notebook code or this chat. See the [official API guide](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=16). Authenticated EVDS downloads have not been tested. The EVDS requests are split by year to stay below the API's 1,000-observation limit.


In [ ]:
import json
import os
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path

import requests
from dotenv import load_dotenv

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
end_date = date.today().replace(day=1) - timedelta(days=1)
load_dotenv(dotenv_path=Path(".env"), override=False)
api_key = os.environ.get("EVDS_API_KEY", "").strip() or getpass("EVDS API key: ").strip()
if not api_key:
    raise ValueError("An EVDS API key is required for Turkish CPI and USD/TRY.")

fao_url = (
    "https://www.fao.org/media/docs/worldfoodsituationlibraries/"
    "wfs-library/food_price_indices_data.csv?download=true&sfvrsn=523ebd2a_84"
)
response = requests.get(fao_url, timeout=90)
response.raise_for_status()
if not response.content or "text/html" in response.headers.get("Content-Type", "").lower():
    raise ValueError("FAO returned no data or a web page instead of a CSV.")
(raw_dir / "fao_food_price_indices.csv").write_bytes(response.content)

series = {
    "turkish_food_cpi": [
        "TP.TUKFIY2025.01",
        "TP.TUKFIY2025.0111",
        "TP.TUKFIY2025.0112",
        "TP.TUKFIY2025.0114",
        "TP.TUKFIY2025.0115",
    ],
    "usd_try_daily": ["TP.DK.USD.A.YTL"],
}
for dataset, codes in series.items():
    for year in range(2005, end_date.year + 1):
        year_end = min(date(year, 12, 31), end_date)
        url = (
            f"https://evds3.tcmb.gov.tr/igmevdsms-dis/series={'-'.join(codes)}"
            f"&startDate=01-01-{year}&endDate={year_end:%d-%m-%Y}&type=json"
        )
        response = requests.get(url, headers={"key": api_key}, timeout=90)
        if response.status_code in {401, 403}:
            raise RuntimeError("EVDS rejected the request. Check your API key and try again.")
        response.raise_for_status()
        payload = response.json()
        records = payload.get("items") if isinstance(payload, dict) else payload
        if not isinstance(records, list) or not records:
            raise ValueError(f"EVDS returned no observations for {dataset}, {year}.")
        (raw_dir / f"{dataset}_{year}.json").write_text(
            json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
        )

print(f"Raw downloads saved in {raw_dir.resolve()}")
